# NeuraRoads - 02 Model Training (from scratch)

Train / monitor the YOLOv8m detector trained **from scratch** (no COCO weights) on the 10-class dataset. Long runs are best launched from the CLI:
```
python src/training/train_yolo.py
```
This notebook lets you launch a short run and inspect the results.

In [ ]:
import sys, os
from pathlib import Path
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
SRC = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
from training.train_yolo import load_hyperparameters
hyp = load_hyperparameters()
print({k: hyp[k] for k in ['architecture','pretrained','epochs','imgsz','batch','optimizer','lr0']})

In [ ]:
# Optional: launch a SHORT training run (e.g. 3 epochs) to smoke-test the setup.
# Remove the small epochs/batch to run the full 150-epoch schedule.
from ultralytics import YOLO
from utils.config_loader import resolve_path
model = YOLO(f"{hyp['architecture']}.yaml")  # random init = from scratch
results = model.train(data=str(resolve_path('data/annotations/data.yaml')),
                      epochs=3, imgsz=640, batch=8, pretrained=False,
                      project=str(resolve_path('models/trained')),
                      name='nb_smoketest', exist_ok=True)

In [ ]:
# Inspect the training curves produced by Ultralytics.
import matplotlib.pyplot as plt, matplotlib.image as mpimg
from utils.config_loader import resolve_path
run = resolve_path('models/trained/nb_smoketest')
png = run / 'results.png'
if png.is_file():
    plt.figure(figsize=(14, 8)); plt.imshow(mpimg.imread(str(png))); plt.axis('off'); plt.show()
else:
    print('Run training first; results.png not found at', png)

In [ ]:
# Validate the best checkpoint and print per-class AP50.
from utils.config_loader import resolve_path
best = resolve_path('models/trained/nb_smoketest/weights/best.pt')
if best.is_file():
    m = YOLO(str(best))
    metrics = m.val(data=str(resolve_path('data/annotations/data.yaml')))
    print('mAP50-95:', float(metrics.box.map), '| mAP50:', float(metrics.box.map50))
else:
    print('No trained weights yet.')